# Triage: planta / animal / otro, y tipo general

Paso previo a la identificación de especie, para las fotos que no tienen ninguna pista de carpeta (ramas `AUTORES` y `EVENTOS`, organizadas por quién tomó la foto o en qué evento, no por especie). Se corre también sobre el resto de fotos sin especie (rama `ESPECIE`) como cruce de validación contra la carpeta original, ya que es gratis hacerlo.

**Por qué CLIP local y no un modelo entrenado a la medida:** no hay datos etiquetados para entrenar algo propio, y no hace falta — esto es clasificación *zero-shot* (el modelo compara la imagen contra frases candidato tipo "a photo of a tree", sin haber visto ejemplos de Campus Lab). Corre en tu máquina, sin costo por imagen ni cuota diaria (a diferencia de Pl@ntNet), así que se puede correr sobre miles de fotos y volver a correr cuantas veces haga falta.

**Dos pasos:**
1. `reino_sugerido`: planta / animal / otro (personas, edificios, eventos sin sujeto biológico).
2. `tipo_sugerido`: si es planta → árbol / suculenta / flor / orquídea / semilla / otra planta. Si es animal → pájaro / insecto / otro animal. Categorías tomadas de las carpetas que ya existen en el volumen y de `BASE_DATOS_GENERAL.xlsx` (hojas `Lista_AVES`, `Lista_ÁRBOLES`, `Lista_SUCULENTAS`), no inventadas.

**Esto no confirma especie, solo enruta.** El resultado (`fotos.reino_sugerido`, `fotos.tipo_sugerido`, con su `tipo_confianza`) sirve para que el siguiente paso decida a qué identificador específico mandar cada foto (Pl@ntNet para plantas, iNaturalist/eBird para pájaros e insectos), en vez de depender solo de la carpeta original. Por eso no se filtra por confianza mínima aquí — un acierto parcial en el triage cuesta mandar la foto al identificador equivocado (que simplemente no la reconoce), no ensucia el catálogo de especies.

**Insumo:** `../../experimentos/bd/campuslab_volumen.db` (`fotos.ruta_actual`, la ruta real, nunca `staging_escaneo.ruta_completa`).
**Salida:** columnas nuevas en `fotos`, nada en `especies` ni en el nombre físico del archivo.

In [ ]:
import sqlite3
from pathlib import Path

import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

RUTA_BD = Path("../../experimentos/bd/campuslab_volumen.db")  # la BD real, sin mover ni duplicar
con = sqlite3.connect(RUTA_BD)
print(f"Base de datos: {RUTA_BD.resolve()} ({'existe' if RUTA_BD.exists() else 'NO ENCONTRADA'})")

MODEL_ID = "openai/clip-vit-base-patch32"
print(f"Cargando {MODEL_ID} (primera vez descarga ~600MB, luego queda en cache local)...")
modelo = CLIPModel.from_pretrained(MODEL_ID)
procesador = CLIPProcessor.from_pretrained(MODEL_ID)
modelo.eval()
print("Modelo listo.")

## 1. Esquema: columnas de triage en `fotos`

In [ ]:
cols_fotos = [r[1] for r in con.execute("PRAGMA table_info(fotos)")]
for col, tipo in [("reino_sugerido", "TEXT"), ("reino_confianza", "REAL"),
                   ("tipo_sugerido", "TEXT"), ("tipo_confianza", "REAL"),
                   ("tipo_sugerido_por", "TEXT")]:
    if col not in cols_fotos:
        con.execute(f"ALTER TABLE fotos ADD COLUMN {col} {tipo}")
con.commit()
print("Esquema listo: fotos.reino_sugerido / reino_confianza / tipo_sugerido / tipo_confianza / tipo_sugerido_por")

## 2. Etiquetas candidato (en inglés: CLIP rinde mejor así) y función de clasificación

In [ ]:
REINOS = [
    {"clave": "planta", "prompt": "a photo of a plant, tree, flower or succulent"},
    {"clave": "animal", "prompt": "a photo of an animal, bird or insect"},
    {"clave": "otro", "prompt": "a photo of people, buildings or objects, with no plant or animal as the main subject"},
]

TIPOS_PLANTA = [
    {"clave": "arbol", "prompt": "a photo of a tree"},
    {"clave": "suculenta", "prompt": "a photo of a succulent or cactus plant"},
    {"clave": "orquidea", "prompt": "a photo of an orchid flower"},
    {"clave": "flor", "prompt": "a photo of a flower that is not an orchid"},
    {"clave": "semilla", "prompt": "a photo of a seed or seed pod"},
    {"clave": "otra_planta", "prompt": "a photo of another kind of plant, leaf, or vegetation"},
]

TIPOS_ANIMAL = [
    {"clave": "pajaro", "prompt": "a photo of a bird"},
    {"clave": "insecto", "prompt": "a photo of an insect or spider"},
    {"clave": "otro_animal", "prompt": "a photo of another kind of animal, such as a mammal or reptile"},
]

EXTENSIONES_SOPORTADAS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

def clasificar_zero_shot(imagen_pil, etiquetas):
    """Devuelve (clave_top, confianza) comparando la imagen contra cada prompt candidato."""
    prompts = [e["prompt"] for e in etiquetas]
    entradas = procesador(text=prompts, images=imagen_pil, return_tensors="pt", padding=True)
    with torch.no_grad():
        salida = modelo(**entradas)
        probs = salida.logits_per_image.softmax(dim=1)[0]
    idx = int(probs.argmax())
    return etiquetas[idx]["clave"], float(probs[idx])

def clasificar_foto(ruta_imagen):
    """Triage completo de una foto: reino, y tipo dentro de ese reino (None si reino='otro')."""
    if Path(ruta_imagen).suffix.lower() not in EXTENSIONES_SOPORTADAS:
        return None
    with Image.open(ruta_imagen) as img:
        img = img.convert("RGB")
        reino, conf_reino = clasificar_zero_shot(img, REINOS)
        if reino == "planta":
            tipo, conf_tipo = clasificar_zero_shot(img, TIPOS_PLANTA)
        elif reino == "animal":
            tipo, conf_tipo = clasificar_zero_shot(img, TIPOS_ANIMAL)
        else:
            tipo, conf_tipo = None, None
    return reino, conf_reino, tipo, conf_tipo

## 3. Prueba rápida contra una foto conocida

Antes de correr todo el lote, valida contra una foto que ya sepamos que es planta (rama `ESPECIE`, breadcrumb `ÁRBOLES`), para confirmar que el modelo y las rutas funcionan.

In [ ]:
fila_prueba = con.execute("""
    SELECT f.id, f.ruta_actual
    FROM fotos f JOIN staging_escaneo s ON s.ruta_completa = f.ruta_original
    WHERE s.breadcrumb_especie LIKE '%RBOLES%' AND f.especie_id IS NULL
    LIMIT 1
""").fetchone()

if fila_prueba:
    foto_id, ruta = fila_prueba
    print(f"Probando con foto {foto_id}: {ruta}")
    resultado = clasificar_foto(ruta)
    print(f"Resultado: {resultado}")
else:
    print("No encontre una foto de prueba en ARBOLES, revisa la consulta.")

## 4. Correr sobre todas las fotos sin especie

Resumible: excluye las que ya tienen `reino_sugerido`. Commit cada 100 fotos para no perder progreso si se interrumpe a medias.

In [ ]:
candidatas = con.execute("""
    SELECT id, ruta_actual FROM fotos
    WHERE especie_id IS NULL AND reino_sugerido IS NULL
""").fetchall()
print(f"Candidatas a clasificar: {len(candidatas):,}")

procesadas = 0
omitidas = 0
errores = 0
for i, (foto_id, ruta) in enumerate(candidatas, 1):
    try:
        resultado = clasificar_foto(ruta)
        if resultado is None:
            omitidas += 1
            con.execute(
                "UPDATE fotos SET reino_sugerido=?, tipo_sugerido_por=? WHERE id=?",
                ("no_soportado", "CLIP", foto_id),
            )
        else:
            reino, conf_reino, tipo, conf_tipo = resultado
            con.execute(
                "UPDATE fotos SET reino_sugerido=?, reino_confianza=?, tipo_sugerido=?, tipo_confianza=?, tipo_sugerido_por=? WHERE id=?",
                (reino, conf_reino, tipo, conf_tipo, "CLIP", foto_id),
            )
            procesadas += 1
    except Exception as e:
        errores += 1
        print(f"Error en foto {foto_id} ({ruta}): {e}")

    if i % 100 == 0:
        con.commit()
        print(f"{i:,} / {len(candidatas):,} (ok={procesadas}, omitidas={omitidas}, errores={errores})")

con.commit()
print(f"\nListo. ok={procesadas}, omitidas(formato no soportado)={omitidas}, errores={errores}")

## 5. Reporte: distribución de reino/tipo, y cruce contra la carpeta original

In [ ]:
print("--- Reino sugerido ---")
for reino, n in con.execute("SELECT reino_sugerido, COUNT(*) FROM fotos WHERE reino_sugerido IS NOT NULL GROUP BY reino_sugerido ORDER BY COUNT(*) DESC"):
    print(f"  {reino}: {n:,}")

print("\n--- Tipo sugerido ---")
for tipo, n in con.execute("SELECT tipo_sugerido, COUNT(*) FROM fotos WHERE tipo_sugerido IS NOT NULL GROUP BY tipo_sugerido ORDER BY COUNT(*) DESC"):
    print(f"  {tipo}: {n:,}")

print("\n--- Cruce: reino sugerido vs. rama de carpeta original (fotos de rama ESPECIE) ---")
q = """
    SELECT s.breadcrumb_especie, f.reino_sugerido, COUNT(*)
    FROM fotos f JOIN staging_escaneo s ON s.ruta_completa = f.ruta_original
    WHERE s.rama_raiz = 'ESPECIE' AND f.reino_sugerido IS NOT NULL
    GROUP BY s.breadcrumb_especie, f.reino_sugerido
    ORDER BY COUNT(*) DESC LIMIT 20
"""
for carpeta, reino, n in con.execute(q):
    print(f"  {carpeta} -> {reino}: {n}")